In [1]:
pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.5 MB/s eta 0:00:00


In [2]:
import qiskit
print(qiskit.__version__)

2.5.2


Import Required Libraries

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix

Exercise 1 Easy - Construct |Φ⁺⟩ Bell State and Verify 1024-Shot Correlation

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Initialize 2-qubit circuit for |Phi+> state
bell_pair_circ = QuantumCircuit(2, 2)
bell_pair_circ.h(0)
bell_pair_circ.cx(0, 1)
bell_pair_circ.measure([0, 1], [0, 1])
backend = AerSimulator()
simulation_result = backend.run(bell_pair_circ, shots=1024).result().get_counts()
print("--- Circuit Topology: |Phi+> State ---")
print(bell_pair_circ)
print("\nSampled Bitstring Statistics (1024 Shots):", simulation_result)
is_valid_pair = all(bitstring in ['00', '11'] for bitstring in simulation_result.keys())
print("Entanglement Check:", "PASSED (Zero probability for '01' and '10')" if is_valid_pair else "FAILED")

--- Circuit Topology: |Phi+> State ---
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Sampled Bitstring Statistics (1024 Shots): {'11': 505, '00': 519}
Entanglement Check: PASSED (Zero probability for '01' and '10')


Exercise 2 Medium -
Construct and Tabulate All Four Bell States (|Φ⁺⟩, |Φ⁻⟩, |Ψ⁺⟩, |Ψ⁻⟩)

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Construct four Bell states via input basis encoding
states_dict = {
    "|Phi+>": lambda qc: None,
    "|Phi->": lambda qc: qc.z(0),
    "|Psi+>": lambda qc: qc.x(1),
    "|Psi->": lambda qc: (qc.x(1), qc.z(0))
}
sim_engine = AerSimulator()
print("--- Characterization of Complete Bell Basis (1000 Shots) ---")
print(f"{'State':<10}{'P(00)':<12}{'P(01)':<12}{'P(10)':<12}{'P(11)':<12}{'Parity Signature'}")
print("-" * 65)
for label, gate_ops in states_dict.items():
    qc_bell = QuantumCircuit(2, 2)
    gate_ops(qc_bell)
    qc_bell.h(0)
    qc_bell.cx(0, 1)
    qc_bell.measure([0, 1], [0, 1])
    counts = sim_engine.run(qc_bell, shots=1000).result().get_counts()
    p00 = counts.get('00', 0) / 1000.0
    p01 = counts.get('01', 0) / 1000.0
    p10 = counts.get('10', 0) / 1000.0
    p11 = counts.get('11', 0) / 1000.0
    parity = "Correlated (Even)" if (p00 + p11 > 0.95) else "Anti-correlated (Odd)"
    print(f"{label:<10}{p00:<12.3f}{p01:<12.3f}{p10:<12.3f}{p11:<12.3f}{parity}")

--- Characterization of Complete Bell Basis (1000 Shots) ---
State     P(00)       P(01)       P(10)       P(11)       Parity Signature
-----------------------------------------------------------------
|Phi+>    0.517       0.000       0.000       0.483       Correlated (Even)
|Phi->    0.485       0.000       0.000       0.515       Correlated (Even)
|Psi+>    0.000       0.478       0.522       0.000       Anti-correlated (Odd)
|Psi->    0.000       0.503       0.497       0.000       Anti-correlated (Odd)


Exercise 3 Hard -
Measure Bell Pairs in the Hadamard (X) Basis

In [7]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def setup_transverse_measurement(is_phase_flipped=False):
    circuit = QuantumCircuit(2, 2)
    circuit.h(0)
    if is_phase_flipped:
        circuit.z(0)
    circuit.cx(0, 1)
    # Rotate measurement axis from Z to X
    circuit.barrier()
    circuit.h(0)
    circuit.h(1)
    circuit.measure([0, 1], [0, 1])
    return circuit
circ_phi_p = setup_transverse_measurement(is_phase_flipped=False)
circ_phi_m = setup_transverse_measurement(is_phase_flipped=True)
out_p = sim_engine.run(circ_phi_p, shots=1000).result().get_counts()
out_m = sim_engine.run(circ_phi_m, shots=1000).result().get_counts()
print("--- Transverse (X-Basis) Measurement on |Phi+> ---")
print(circ_phi_p)
print("Readout Distribution:", out_p)
print("\n--- Transverse (X-Basis) Measurement on |Phi-> ---")
print(circ_phi_m)
print("Readout Distribution:", out_m)

--- Transverse (X-Basis) Measurement on |Phi+> ---
     ┌───┐      ░ ┌───┐┌─┐   
q_0: ┤ H ├──■───░─┤ H ├┤M├───
     └───┘┌─┴─┐ ░ ├───┤└╥┘┌─┐
q_1: ─────┤ X ├─░─┤ H ├─╫─┤M├
          └───┘ ░ └───┘ ║ └╥┘
c: 2/═══════════════════╩══╩═
                        0  1 
Readout Distribution: {'11': 475, '00': 525}

--- Transverse (X-Basis) Measurement on |Phi-> ---
     ┌───┐┌───┐      ░ ┌───┐┌─┐   
q_0: ┤ H ├┤ Z ├──■───░─┤ H ├┤M├───
     └───┘└───┘┌─┴─┐ ░ ├───┤└╥┘┌─┐
q_1: ──────────┤ X ├─░─┤ H ├─╫─┤M├
               └───┘ ░ └───┘ ║ └╥┘
c: 2/════════════════════════╩══╩═
                             0  1 
Readout Distribution: {'01': 518, '10': 482}


Exercise 4 Real-world -
Entanglement-Based Quantum Key Distribution (Ekert-91 Protocol Concept)

In [8]:
# Conceptual Diagram: E91 Quantum Key Distribution via Entangled Bell Pairs
# Alice and Bob receive entangled particles from an EPR source.
# Random basis choices detect eavesdropping through Bell inequality violations.
protocol_flow = [
    "--- Architecture of the Entanglement-Based E91 Protocol ---",
    "",
    "   [ Alice's Station ] <=== (|Phi+> Pairs) ===> [ Bob's Station ]",
    "            |                                          |",
    "    Random Basis Select                        Random Basis Select",
    "    {a1, a2, a3}                               {b1, b2, b3}",
    "            |                                          |",
    "    Binary Key String                          Binary Key String",
    "            |                                          |",
    "            +================ Public Channel ==========+",
    "",
    "Operational Stages:",
    "  * Key Sifting: Matching basis orientations generate correlated raw key bits.",
    "  * Quantum Auditing: Non-matching basis orientations evaluate the CHSH parameter S.",
    "    - Noise-free Quantum Channel: S = 2*sqrt(2) approx 2.828 (Violates classical limits).",
    "    - Eavesdropping Interception: Forces state collapse, reducing S <= 2 (Reveals Eve)."
]
print("\n".join(protocol_flow))

--- Architecture of the Entanglement-Based E91 Protocol ---

   [ Alice's Station ] <=== (|Phi+> Pairs) ===> [ Bob's Station ]
            |                                          |
    Random Basis Select                        Random Basis Select
    {a1, a2, a3}                               {b1, b2, b3}
            |                                          |
    Binary Key String                          Binary Key String
            |                                          |
            +================ Public Channel ==========+

Operational Stages:
  * Key Sifting: Matching basis orientations generate correlated raw key bits.
  * Quantum Auditing: Non-matching basis orientations evaluate the CHSH parameter S.
    - Noise-free Quantum Channel: S = 2*sqrt(2) approx 2.828 (Violates classical limits).
    - Eavesdropping Interception: Forces state collapse, reducing S <= 2 (Reveals Eve).


Exercise 5 Challenge -
3-Qubit GHZ vs. W-State: Robustness Under Partial Measurement

In [9]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
# 1. Synthesize 3-qubit GHZ state
ghz_circuit = QuantumCircuit(3)
ghz_circuit.h(0)
ghz_circuit.cx(0, 1)
ghz_circuit.cx(1, 2)
ghz_vector = Statevector.from_instruction(ghz_circuit)
# 2. Synthesize 3-qubit W state
w_circuit = QuantumCircuit(3)
w_circuit.ry(2 * np.arccos(1 / np.sqrt(3)), 0)
w_circuit.ch(0, 1)
w_circuit.cx(1, 2)
w_circuit.cx(0, 1)
w_circuit.x(0)
w_vector = Statevector.from_instruction(w_circuit)
# Partial measurement on qubit 0
readout_ghz, post_ghz = ghz_vector.measure([0])
readout_w, post_w = w_vector.measure([0])
print("--- Tripartite Entanglement Decoherence Analysis ---")
print("Initial GHZ Amplitudes: ", np.round(ghz_vector.data, 3))
print(f"GHZ Measured Qubit 0:   {readout_ghz}")
print("Collapsed GHZ State:    ", np.round(post_ghz.data, 3))
print("\nInitial W State Amplitudes:", np.round(w_vector.data, 3))
print(f"W State Measured Qubit 0:  {readout_w}")
print("Collapsed W State:         ", np.round(post_w.data, 3))

--- Tripartite Entanglement Decoherence Analysis ---
Initial GHZ Amplitudes:  [0.707+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
 0.707+0.j]
GHZ Measured Qubit 0:   0
Collapsed GHZ State:     [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Initial W State Amplitudes: [0.   +0.j 0.577+0.j 0.577+0.j 0.   +0.j 0.577+0.j 0.   +0.j 0.   +0.j
 0.   +0.j]
W State Measured Qubit 0:  0
Collapsed W State:          [0.   +0.j 0.   +0.j 0.707+0.j 0.   +0.j 0.707+0.j 0.   +0.j 0.   +0.j
 0.   +0.j]
